In [25]:
!pip install -q groq

## The Setup

In [26]:
import os
from groq import Groq
from kaggle_secrets import UserSecretsClient

# Access the secret key from Kaggle's vault
user_secrets = UserSecretsClient()
groq_key = user_secrets.get_secret("Groq")

client = Groq(api_key=groq_key)

## Define Your Experts

In [27]:
# Use the newer Llama 3.3 model
CURRENT_MODEL = "llama-3.3-70b-versatile" 

EXPERTS = {
    "technical": {
        "system_prompt": "You are a Senior Software Engineer. Provide rigorous, code-focused solutions.",
        "model": CURRENT_MODEL
    },
    "billing": {
        "system_prompt": "You are a Billing Specialist. Focus on refund policies and transaction empathy.",
        "model": CURRENT_MODEL
    },
    "general": {
        "system_prompt": "You are a friendly general assistant.",
        "model": CURRENT_MODEL
    }
}

## The Router (The "Brain")

In [28]:
def route_prompt(user_input):
    routing_prompt = f"Classify this into [technical, billing, general]. Return ONLY the word: {user_input}"
    
    response = client.chat.completions.create(
        messages=[{"role": "user", "content": routing_prompt}],
        # Using the 8b model here makes the routing nearly instant
        model="llama-3.1-8b-instant", 
        temperature=0
    )
    return response.choices[0].message.content.strip().lower()

## The Orchestrator

In [29]:
def process_request(user_input):
    category = route_prompt(user_input)
    print(f"--- Logic: Routing to {category.upper()} expert ---")
    
    expert_config = EXPERTS.get(category, EXPERTS["general"])
    
    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": expert_config["system_prompt"]},
            {"role": "user", "content": user_input}
        ],
        model=expert_config["model"],
        temperature=0.7
    )
    return response.choices[0].message.content

## Testing

In [30]:
print(process_request("My python script is throwing an IndexError on line 5."))

--- Logic: Routing to GENERAL expert ---
When a Python script throws an `IndexError`, it usually means that you're trying to access an element in a list (or other sequence type) using an index that doesn't exist.

To help you troubleshoot this issue, could you please provide the following details:

1. The code snippet that includes line 5 (the line causing the error)
2. Any relevant data or inputs that might be contributing to the error
3. The full error message you're seeing, including the `IndexError` message

With this information, I'll do my best to help you identify the cause of the issue and suggest a solution.


In [31]:
print(process_request("I was charged twice for my subscription this month."))

--- Logic: Routing to BILLING expert ---
I'm so sorry to hear that you were charged twice for your subscription. I can imagine how frustrating that must be for you. 

Let me see what I can do to help resolve this issue for you. Can you please provide me with your subscription details, such as your account name, subscription plan, and the dates of the duplicate charges? This will help me to locate the issue and process a refund for the incorrect charge as soon as possible.

Additionally, I'd like to assure you that we take situations like this very seriously and are committed to making things right. We have a clear refund policy in place, which states that if a customer is charged in error, we will issue a full refund for the incorrect charge within 3-5 business days.

Please know that I'm here to help and will do my best to resolve this issue promptly and efficiently. If you have any questions or concerns, please don't hesitate to reach out. Your satisfaction is our top priority, and I